# Download a Zillow research data file

This notebook makes one HTTP `GET` request to `files.zillowstatic.com`, saves the response locally, and loads a CSV into pandas. Change `DATA_URL` to another direct Zillow Static file URL when needed.

> Use public files only, and follow Zillow's terms, access rules, and rate limits.

In [0]:
# Run once per environment; pip skips packages that are already satisfied.
%pip install -q requests pandas

from io import BytesIO
from pathlib import Path
from urllib.parse import unquote, urlparse

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## Configure the download

The example is Zillow's county-level, monthly Zillow Observed Rent Index (ZORI) CSV.

In [0]:
DATA_URL = (
    "https://files.zillowstatic.com/research/public_csvs/zori/"
    "County_zori_uc_sfrcondomfr_sm_month.csv"
)
OUTPUT_DIR = Path("data/raw")
TIMEOUT_SECONDS = 60

## Make the HTTP GET request

In [0]:
parsed_url = urlparse(DATA_URL)
if parsed_url.scheme != "https" or parsed_url.hostname != "files.zillowstatic.com":
    raise ValueError("DATA_URL must be an HTTPS URL hosted by files.zillowstatic.com")

headers = {
    "Accept": "text/csv,application/octet-stream;q=0.9,*/*;q=0.1",
    "User-Agent": "rental-analysis/1.0 (one-file research download)",
}
retry_policy = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset({"GET"}),
    respect_retry_after_header=True,
)

with requests.Session() as session:
    session.mount("https://", HTTPAdapter(max_retries=retry_policy))
    response = session.get(
        DATA_URL, headers=headers, timeout=TIMEOUT_SECONDS
    )
response.raise_for_status()

print(f"Status: {response.status_code}")
print(f"Content type: {response.headers.get('content-type', 'unknown')}")
print(f"Downloaded: {len(response.content) / 1_000_000:,.2f} MB")

## Save the file

In [0]:
filename = Path(unquote(parsed_url.path)).name
if not filename:
    raise ValueError("The URL does not contain a filename")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / filename
output_path.write_bytes(response.content)
print(f"Saved to: {output_path.resolve()}")

## Load and inspect the CSV

In [0]:
data = pd.read_csv(BytesIO(response.content))
print(f"Rows: {len(data):,}; columns: {len(data.columns):,}")
data.head()

## Optional: convert monthly columns to tidy rows

Zillow time-series CSVs are usually wide: one column per month. This cell creates one row per geography and month.

In [0]:
date_columns = [
    column
    for column in data.columns
    if pd.notna(pd.to_datetime(str(column), format="%Y-%m-%d", errors="coerce"))
]
id_columns = [column for column in data.columns if column not in date_columns]

rent_long = data.melt(
    id_vars=id_columns,
    value_vars=date_columns,
    var_name="date",
    value_name="zori",
)
rent_long["date"] = pd.to_datetime(rent_long["date"])
rent_long = rent_long.dropna(subset=["zori"]).reset_index(drop=True)

print(f"Tidy rows: {len(rent_long):,}")
rent_long.head()